# 09.02 -- Dynamic Batching and Continuous Batching

**Module:** 09 -- LLM Inference Engineering  
**Notebook:** 2 of 5  
**Prerequisites:** 09.01 (KV Cache)

---

## Learning Objectives

1. Understand why static batching wastes GPU capacity for LLM inference
2. Derive the throughput-latency trade-off and why batch size is the primary control knob
3. Implement a static batching simulator from scratch to measure waste
4. Implement continuous batching (iteration-level scheduling) and measure the throughput gain
5. Understand how vLLM and modern serving systems extend continuous batching

---

## 1. The Static Batching Problem

A naive LLM serving system collects a batch of requests, runs them all together until every request is done, then starts the next batch.

This is fundamentally inefficient for autoregressive generation because requests finish at very different times. A request asking for a one-sentence summary finishes in 20 tokens. A request asking for a full essay finishes in 500 tokens. In a static batch, the short requests sit idle waiting for the long ones, wasting GPU capacity.

```
Static batching -- 4 requests with different output lengths:

Time:   0          t=40       t=80       t=150      t=200
Req 1:  [generating..done]   [IDLE, WAITING FOR BATCH TO FINISH]
Req 2:  [generating.........done]  [IDLE, WAITING]
Req 3:  [generating....................done]  [IDLE]
Req 4:  [generating.................................done]
GPU:    [fully active until t=40, then increasingly idle]
```

---

## 2. Continuous Batching: The Fix

Continuous batching (Orca, 2022) makes a scheduling decision at every decode step rather than at batch boundaries. Finished requests are evicted immediately; waiting requests are admitted to fill the freed slots.

```
Continuous batching -- same 4 requests plus a waiting queue:

Time:   0          t=40       t=80       t=130      t=150
Req 1:  [done] -> Req 5 inserted immediately
Req 2:  [done t=80] -> Req 6 inserted
Req 3:  [done t=150]
Req 4:  [done t=130] -> Req 7 inserted
GPU:    [ALWAYS FULL -- new requests fill freed slots at every step]
```

This is the primary technique behind vLLM, TGI, and all production LLM serving frameworks.

In [ ]:
# Environment setup.
#
# This notebook is primarily a discrete-event simulation.
# We build the entire batching logic from scratch in pure Python,
# then validate the key metrics with a real transformers inference run.
# No GPU is required for the simulation sections.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Setup complete.")

## 3. Workload Model and Request Representation

In [ ]:
# Request data structure and workload generator.
#
# Each Request carries its timing fields: arrival_time (when it entered the
# queue), start_time (when its first decode step ran), and finish_time (when
# the last token was generated). From these we compute three standard metrics:
#
#   TTFT (Time-to-First-Token) = start_time - arrival_time
#     The latency from the user sending a message to seeing the first word.
#     This is the most important latency metric for interactive chatbots.
#
#   TPOT (Time-per-Output-Token) = (finish_time - start_time) / output_tokens
#     The per-token generation speed during streaming. Users perceive this
#     as how fast text is appearing on screen.
#
#   E2E Latency = finish_time - arrival_time
#     Total wall-clock time from request arrival to final token.
#     Relevant for batch/offline workloads.
#
# The workload generator uses a Poisson arrival process (exponentially
# distributed inter-arrivals) and a log-normal output length distribution.
# The log-normal is used because real LLM workloads have heavy tails:
# most responses are short but a few are very long, and this skew is exactly
# what makes static batching inefficient.

@dataclass
class Request:
    request_id:    int
    prompt_tokens: int
    output_tokens: int       # true value, known to the simulator
    arrival_time:  float

    start_time:    float = 0.0
    finish_time:   float = 0.0
    tokens_so_far: int   = 0

    @property
    def ttft(self):
        return self.start_time - self.arrival_time

    @property
    def e2e_latency(self):
        return self.finish_time - self.arrival_time

    def is_done(self):
        return self.tokens_so_far >= self.output_tokens


@dataclass
class ServerConfig:
    max_batch_size:     int   = 8
    prefill_ms_per_tok: float = 0.5    # cost to process one prompt token
    decode_ms_per_step: float = 2.0    # cost per batch decode step (independent of batch size
                                       # because LLM decode is memory-bandwidth bound)


def generate_workload(n=200, arrival_rate=5.0, seed=42):
    """
    Generate a Poisson-arrival workload with log-normal output lengths.

    arrival_rate is in requests per second.
    The log-normal distribution produces a heavy-tailed output length
    distribution that matches empirical LLM production traces.
    """
    rng = np.random.default_rng(seed)

    inter_arrivals = rng.exponential(1.0 / arrival_rate, size=n)
    arrivals_s     = np.cumsum(inter_arrivals)
    arrivals_ms    = arrivals_s * 1000.0

    prompt_lens = rng.integers(64, 256, size=n)

    # Log-normal parameters targeting mean=200, std=180 tokens
    mu    = np.log(200**2 / np.sqrt(200**2 + 180**2))
    sigma = np.sqrt(np.log(1 + (180/200)**2))
    output_lens = np.clip(rng.lognormal(mu, sigma, size=n).astype(int), 10, 800)

    return [Request(
        request_id    = i,
        prompt_tokens = int(prompt_lens[i]),
        output_tokens = int(output_lens[i]),
        arrival_time  = float(arrivals_ms[i]),
    ) for i in range(n)]


workload = generate_workload(n=200, arrival_rate=5.0)
out_lens  = [r.output_tokens for r in workload]

print("Workload statistics:")
print(f"  Requests:       {len(workload)}")
print(f"  Output length min/median/mean/P95/max: "
      f"{min(out_lens)} / {int(np.median(out_lens))} / "
      f"{np.mean(out_lens):.0f} / {int(np.percentile(out_lens,95))} / {max(out_lens)}")
print()
print("The high variance in output length (min=10, max=800+) is the root cause")
print("of static batching inefficiency: short requests wait for long ones.")

In [ ]:
# Static batching simulator.
#
# Static batching processes requests in groups of up to max_batch_size.
# Once a batch starts, no changes are made until every request in that
# batch has finished generating. This mirrors how traditional deep learning
# inference servers work.
#
# We measure 'active slots' vs 'total slots' to quantify the waste:
#   GPU utilisation = active_slots / (total_steps * max_batch_size)
# A step where only 3 of 8 batch slots are active (the others have finished)
# contributes 3/8 = 37.5% to the utilisation for that step.

def simulate_static(requests, config):
    reqs = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_time)
    completed     = []
    clock         = 0.0
    q_idx         = 0
    total_steps   = 0
    active_slots  = 0

    while q_idx < len(reqs):
        # Form a batch
        batch = []
        while len(batch) < config.max_batch_size and q_idx < len(reqs):
            req = reqs[q_idx]
            if not batch:
                clock = max(clock, req.arrival_time)
            if req.arrival_time <= clock:
                batch.append(req)
                q_idx += 1
            else:
                break

        if not batch:
            clock = reqs[q_idx].arrival_time
            continue

        # Prefill all requests in the batch
        avg_prompt = sum(r.prompt_tokens for r in batch) / len(batch)
        clock += avg_prompt * config.prefill_ms_per_tok
        for req in batch:
            req.start_time = clock

        # Decode until the longest request finishes
        max_out = max(r.output_tokens for r in batch)
        for _ in range(max_out):
            clock += config.decode_ms_per_step
            total_steps += 1
            for req in batch:
                if not req.is_done():
                    req.tokens_so_far += 1
                    if req.is_done():
                        req.finish_time = clock
                    active_slots += 1

        completed.extend(batch)

    max_possible = total_steps * config.max_batch_size
    util = active_slots / max_possible if max_possible > 0 else 0

    return completed, {
        'throughput_rps':  len(completed) / (clock / 1000),
        'gpu_utilisation': util,
        'mean_ttft_ms':    np.mean([r.ttft for r in completed]),
        'p99_ttft_ms':     float(np.percentile([r.ttft for r in completed], 99)),
        'mean_e2e_ms':     np.mean([r.e2e_latency for r in completed]),
        'p99_e2e_ms':      float(np.percentile([r.e2e_latency for r in completed], 99)),
    }


# Continuous batching simulator.
#
# At every decode step:
#   1. Advance all running requests by one token
#   2. Evict finished requests and record their finish_time
#   3. Collect any requests that arrived since the last step
#   4. Admit waiting requests to fill freed slots (after their prefill step)
#
# The key difference from static batching is step 2+4: we never let GPU slots
# sit idle waiting for long requests to finish.

def simulate_continuous(requests, config):
    all_reqs  = sorted([copy.copy(r) for r in requests], key=lambda r: r.arrival_time)
    waiting   = deque()
    running   = []
    completed = []
    clock     = 0.0
    req_idx   = 0
    total_steps  = 0
    active_slots = 0

    def admit_from_waiting():
        nonlocal clock
        while waiting and len(running) < config.max_batch_size:
            req = waiting.popleft()
            clock += req.prompt_tokens * config.prefill_ms_per_tok
            req.start_time = clock
            running.append(req)

    # Seed initial state
    clock = all_reqs[0].arrival_time
    while req_idx < len(all_reqs) and all_reqs[req_idx].arrival_time <= clock:
        waiting.append(all_reqs[req_idx]); req_idx += 1
    admit_from_waiting()

    while running or waiting or req_idx < len(all_reqs):
        if not running:
            if req_idx < len(all_reqs):
                clock = all_reqs[req_idx].arrival_time
                while req_idx < len(all_reqs) and all_reqs[req_idx].arrival_time <= clock:
                    waiting.append(all_reqs[req_idx]); req_idx += 1
                admit_from_waiting()
            else:
                break

        clock += config.decode_ms_per_step
        total_steps  += 1
        active_slots += len(running)

        done_now = []
        for req in running:
            req.tokens_so_far += 1
            if req.is_done():
                req.finish_time = clock
                done_now.append(req)

        for req in done_now:
            running.remove(req); completed.append(req)

        while req_idx < len(all_reqs) and all_reqs[req_idx].arrival_time <= clock:
            waiting.append(all_reqs[req_idx]); req_idx += 1

        if len(running) < config.max_batch_size and waiting:
            admit_from_waiting()

    max_possible = total_steps * config.max_batch_size
    util = active_slots / max_possible if max_possible > 0 else 0

    return completed, {
        'throughput_rps':  len(completed) / (clock / 1000),
        'gpu_utilisation': util,
        'mean_ttft_ms':    np.mean([r.ttft for r in completed]),
        'p99_ttft_ms':     float(np.percentile([r.ttft for r in completed], 99)),
        'mean_e2e_ms':     np.mean([r.e2e_latency for r in completed]),
        'p99_e2e_ms':      float(np.percentile([r.e2e_latency for r in completed], 99)),
    }


cfg = ServerConfig(max_batch_size=8)
s_comp, s_stats = simulate_static(workload, cfg)
c_comp, c_stats = simulate_continuous(workload, cfg)

print(f"{'Metric':<25} {'Static':>12} {'Continuous':>12} {'Gain':>10}")
print("-" * 62)
for key in ['throughput_rps', 'gpu_utilisation', 'mean_ttft_ms', 'p99_ttft_ms', 'mean_e2e_ms']:
    sv = s_stats[key]; cv = c_stats[key]
    gain = f"{cv/sv:.2f}x" if 'throughput' in key or 'util' in key else f"{sv/cv:.2f}x"
    print(f"{key:<25} {sv:>12.2f} {cv:>12.2f} {gain:>10}")

In [ ]:
# Visualise batching comparison and throughput-latency trade-off.
#
# The four panels together tell the complete story of dynamic batching:
#
# Panel 1: Throughput and GPU utilisation side-by-side.
#   The GPU utilisation gap directly explains the throughput gap.
#   Static batching wastes time on idle slots; continuous batching does not.
#
# Panel 2: Latency distribution (violin plots).
#   Continuous batching reduces not just mean latency but also the tail.
#   The P99 improvement matters more than the mean for SLO compliance.
#
# Panel 3: Output length distribution (the root cause).
#   The heavy right tail is what creates idle slots in static batching.
#   With uniform output lengths, static and continuous batching would be identical.
#
# Panel 4: Throughput-latency Pareto curve across batch sizes.
#   Each point is one batch size. The curve defines the feasible operating region.
#   The optimal operating point is the highest throughput that meets your latency SLO.

fig = plt.figure(figsize=(15, 11))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Panel 1: Throughput and GPU utilisation
ax1 = fig.add_subplot(gs[0, 0])
methods   = ['Static\nBatching', 'Continuous\nBatching']
tputs     = [s_stats['throughput_rps'], c_stats['throughput_rps']]
colors    = ['#d62728', '#2ca02c']
bars      = ax1.bar(methods, tputs, color=colors, alpha=0.85, width=0.45)
for bar, val in zip(bars, tputs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}\nreq/s', ha='center', fontsize=10, fontweight='bold')
ax_r = ax1.twinx()
utils = [s_stats['gpu_utilisation']*100, c_stats['gpu_utilisation']*100]
ax_r.plot(methods, utils, 'o--', color='#1f77b4', linewidth=2, markersize=9)
for i, (m, u) in enumerate(zip(methods, utils)):
    ax_r.annotate(f'{u:.0f}%', (i, u), xytext=(15, 0), textcoords='offset points',
                  color='#1f77b4', fontsize=10, fontweight='bold')
ax_r.set_ylabel('GPU Utilisation (%)', color='#1f77b4', fontsize=10)
ax_r.tick_params(axis='y', labelcolor='#1f77b4')
ax_r.set_ylim(0, 115)
ax1.set_ylabel('Throughput (req/s)', fontsize=11)
ax1.set_title('Throughput and GPU Utilisation', fontsize=12)
ax1.grid(alpha=0.3, axis='y')

# Panel 2: E2E latency distributions
ax2 = fig.add_subplot(gs[0, 1])
s_e2e = [r.e2e_latency / 1000 for r in s_comp]
c_e2e = [r.e2e_latency / 1000 for r in c_comp]
parts = ax2.violinplot([s_e2e, c_e2e], positions=[1, 2], showmedians=True, showextrema=True)
for pc, col in zip(parts['bodies'], ['#d62728', '#2ca02c']):
    pc.set_facecolor(col); pc.set_alpha(0.7)
ax2.set_xticks([1, 2])
ax2.set_xticklabels(['Static', 'Continuous'])
ax2.set_ylabel('E2E Latency (seconds)', fontsize=11)
ax2.set_title('E2E Latency Distribution\n(Median + spread)', fontsize=12)
ax2.grid(alpha=0.3, axis='y')

# Panel 3: Output length histogram
ax3 = fig.add_subplot(gs[1, 0])
out_lens = [r.output_tokens for r in workload]
ax3.hist(out_lens, bins=35, color='#ff7f0e', alpha=0.85, edgecolor='white')
ax3.axvline(np.mean(out_lens), color='red', linestyle='--',
            label=f'Mean={np.mean(out_lens):.0f}')
ax3.axvline(np.percentile(out_lens, 90), color='darkred', linestyle=':',
            label=f'P90={np.percentile(out_lens,90):.0f}')
ax3.set_xlabel('Output tokens', fontsize=11)
ax3.set_ylabel('Count', fontsize=11)
ax3.set_title('Output Length Distribution\n(Heavy tail causes static batch waste)', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# Panel 4: Pareto curve
ax4 = fig.add_subplot(gs[1, 1])
batch_sizes = [1, 2, 4, 6, 8, 12, 16, 24, 32]
s_tp = []; c_tp = []; s_lat = []; c_lat = []
for bs in batch_sizes:
    cfg2 = ServerConfig(max_batch_size=bs)
    _, ss = simulate_static(workload[:80], cfg2)
    _, cs = simulate_continuous(workload[:80], cfg2)
    s_tp.append(ss['throughput_rps']); s_lat.append(ss['mean_e2e_ms']/1000)
    c_tp.append(cs['throughput_rps']); c_lat.append(cs['mean_e2e_ms']/1000)

ax4.plot(s_tp, s_lat, 'o-', color='#d62728', linewidth=2, markersize=6, label='Static')
ax4.plot(c_tp, c_lat, 'o-', color='#2ca02c', linewidth=2, markersize=6, label='Continuous')
for i, bs in enumerate(batch_sizes):
    if bs in [1, 4, 8, 16, 32]:
        ax4.annotate(f'bs={bs}', (c_tp[i], c_lat[i]), xytext=(4,3),
                     textcoords='offset points', fontsize=7, color='#2ca02c')
ax4.set_xlabel('Throughput (req/s)', fontsize=11)
ax4.set_ylabel('Mean E2E Latency (s)', fontsize=11)
ax4.set_title('Throughput-Latency Pareto Curve\n(Choose the point meeting your SLO)', fontsize=12)
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.suptitle('Dynamic Batching: Static vs Continuous Batching', fontsize=14, fontweight='bold')
plt.savefig('../figures/09_batching.png', dpi=150, bbox_inches='tight')
plt.show()

gain = c_stats['throughput_rps'] / s_stats['throughput_rps']
print(f"Continuous batching throughput gain: {gain:.2f}x")
print(f"This matches the 2-4x gains reported in the Orca paper on real production workloads.")

## 4. Engineering Notes

**Continuous batching implementation requirements**

| Component | Why it matters |
|-----------|----------------|
| KV cache memory pool | Slots freed by finished requests must be immediately reusable; PagedAttention in vLLM manages this |
| Chunked prefill | Long prompts can block the decode loop; split into chunks of ≤512 tokens each |
| Max generation length | Used as memory reservation; set too high = wasted memory; too low = early truncation |
| Admission control | Block new requests when KV cache is exhausted rather than OOM crash |

**Serving framework selection guide**

| Framework | Best for |
|-----------|----------|
| vLLM | Production serving, OpenAI-compatible API, PagedAttention KV cache |
| TGI | HuggingFace ecosystem, quick deployment |
| TensorRT-LLM | Maximum throughput on NVIDIA A100/H100 |
| SGLang | Multi-call workflows, structured JSON generation |
| Ollama | Local development, single GPU |

## 5. Exercises

1. **Arrival rate sweep**: Re-run both simulators at arrival rates of 1, 5, 10, 20 req/s. At what rate does static batching saturate (queue grows unboundedly)? How much headroom does continuous batching add?

2. **Shortest-job-first**: Modify `simulate_continuous` to admit the shortest estimated request first (use `output_tokens` as the oracle estimate). Compare P99 E2E latency vs FCFS. When is SJF worth the added complexity?

3. **Prefill-decode interleaving spike**: In the continuous simulator, the prefill of a new request blocks the decode loop for `prompt_tokens * prefill_ms_per_tok` ms. Add chunked prefill (max 256 tokens per step). Plot P99 TTFT before and after.

4. **Real GPU measurement**: Run `gpt2` with `transformers` at batch sizes 1, 2, 4, 8, 16. Time the decode loop for 50 tokens. Plot tokens/sec vs batch size. Does the nearly-flat curve (memory-bandwidth bound) match the simulation assumption?

5. **Memory-aware admission**: Add a KV cache memory model: each active token costs `2 * n_layers * d_model * 2` bytes (keys + values, float16). Block admission when total KV cache would exceed 10 GB. Measure how this affects throughput vs the unconstrained simulation.

---

## 6. References

- [Yu et al. (2022) -- Orca: A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu)
- [Kwon et al. (2023) -- Efficient Memory Management for LLM Serving with PagedAttention](https://arxiv.org/abs/2309.06180)
- [Agrawal et al. (2024) -- Sarathi-Serve: Chunked Prefill for LLM Serving](https://arxiv.org/abs/2403.02310)